In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import json
import glob
from pathlib import Path
import torch

# Add project root to path
project_root = "/mnt/caiw_repos/AttnLRP"
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
bench_name, traces_dir = "IFEval", Path("/mnt/caiw/exp/IFEval/data/traces") # IFEval
# bench_name, traces_dir = "HumanEval",  Path("/mnt/caiw/exp/evalplus/data/traces") # HumanEval
# bench_name, traces_dir = "MATH",  Path("/mnt/caiw/exp/MATH/qwen3-0.6b/traces") # MATH
# bench_name, traces_dir = "MMLU_Pro",  Path("/mnt/caiw/exp/MMLU_Pro/data/test_data/traces") # MMLU_Pro

In [ ]:
import pandas as pd
from tqdm import tqdm

json_files = list(traces_dir.glob("*.json"))
print(f"Found {len(json_files)} json files.")

all_records = []
excluded_files = []

# Iterate over all found json files
for file_path in tqdm(json_files, desc="Processing files"):
    start_len = len(all_records)
    
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Failed to load {file_path}: {e}")
        excluded_files.append(file_path.name)
        continue
        
    # Helper to extract data
    def extract_rollouts(source_data, key, model_label):
        rollouts = source_data.get(key)
        if not rollouts:
            return

        # rollouts should be a list of top-k candidates
        for idx, item in enumerate(rollouts):
            # Try to get specific fields, robustly
            record = {
                "sample_id": data['metadata']['uuid'],
                "filename": file_path.name,
                "model": model_label,
                "rank": item.get("rank", idx), 
                "token_str": item.get("token_str"),
                "token_id": item.get("token_id"),
                "logits": item.get("logit", item.get("logits")),
                "eval_result": item.get("eval_result"),
                "completion": item.get("completion", "")
            }
            all_records.append(record)

    # 2. Extract "topk_token_explore" -> model="current"
    extract_rollouts(data, "topk_token_explore", "current")
    
    # Extract "1.7b/7b_topk_token_explore" -> model="7b"
    if "1.7b_topk_token_explore" in data:
        extract_rollouts(data, "1.7b_topk_token_explore", "1.7b")
    elif "7b_topk_token_explore" in data:
        extract_rollouts(data, "7b_topk_token_explore", "1.7b")
    
    if len(all_records) == start_len:
        excluded_files.append(file_path.name)

# 4. Create DataFrame
df = pd.DataFrame(all_records)
if not df.empty:
    df = df.sort_values(by=['sample_id', 'model', 'rank']).reset_index(drop=True)

# Display result info
print(f"Total records: {len(df)}")
if not df.empty:
    print("Unique samples:", df['sample_id'].nunique())

if excluded_files:
    print(f"\nFiles not captured in df ({len(excluded_files)}):")
    print(", ".join(sorted(excluded_files)))
else:
    print("\nAll files were captured in df.")

df.head()

In [ ]:
# ---------------------------------------------------------
# Filter 1: Current model top-1 eval_result check
# Rule: Top-1 prediction of 'current' model must be False (Failed)
# ---------------------------------------------------------

# Find top-1 for current model
current_model_df = df[df['model'] == 'current']
# Sort by rank to ensure we get the top prediction
top1_preds = current_model_df.sort_values(['sample_id', 'rank']).groupby('sample_id').head(1)

# Identify samples where eval_result is NOT False (i.e. True)
violating_samples_1 = top1_preds[top1_preds['eval_result'] != False]['sample_id'].values

print(f"Check 1: Found {len(violating_samples_1)} samples where current model top-1 eval_result != False.")
if len(violating_samples_1) > 0:
    print("Dropping these samples:", violating_samples_1)
    df = df[~df['sample_id'].isin(violating_samples_1)]

# ---------------------------------------------------------
# Filter 2: Completeness check
# Rule: Samples must have both 'current' and '1.7b' models
# ---------------------------------------------------------

models_per_sample = df.groupby('sample_id')['model'].unique().apply(set)
required_models = {'current', '1.7b'}
violating_samples_2 = models_per_sample[models_per_sample != required_models].index.tolist()

print(f"\nCheck 2: Found {len(violating_samples_2)} samples missing required models {required_models}.")
if len(violating_samples_2) > 0:
    print("Dropping these samples:", violating_samples_2)
    df = df[~df['sample_id'].isin(violating_samples_2)]

# Reset index after filtering
df = df.sort_values(by=['sample_id', 'model', 'rank']).reset_index(drop=True)

print(f"\nFinal cleaned dataset: {len(df)} records, {df['sample_id'].nunique()} unique samples.")

In [ ]:
def get_contrast_token_info(group):
    # 1. Search in 'current' model
    current_candidates = group[group['model'] == 'current'].sort_values('rank')
    for _, row in current_candidates.iterrows():
        if row.get('eval_result') is True:
            return pd.Series({
                'contrast_token': row['token_str'],
                'contrast_token_id': int(row['token_id']),
                'extraction_source': f"current_{row['rank']}"
            })
            
    # 2. Search in '1.7b' model
    ref_candidates = group[group['model'] == '1.7b'].sort_values('rank')
    for _, row in ref_candidates.iterrows():
        if row.get('eval_result') is True:
            return pd.Series({
                'contrast_token': row['token_str'],
                'contrast_token_id': int(row['token_id']),
                'extraction_source': f"1.7b_{row['rank']}"
            })
            
    # 3. Not found
    return pd.Series({
        'contrast_token': 'none',
        'contrast_token_id': -1,
        'extraction_source': 'none'
    })

# Apply to each sample
contrast_df = df.groupby('sample_id').apply(get_contrast_token_info).reset_index()

print("Extraction Source Distribution:")
print(contrast_df['extraction_source'].value_counts())

contrast_df.head()

In [ ]:
from attnlrp_circuit.backend.models.manager import ModelManager
from attnlrp_circuit.backend.core import AttributionEngine
import torch

# Initialize backend
# Note: This might take a while to load the first model
print("Initializing ModelManager...")
model_manager = ModelManager()
attribution_engine = AttributionEngine(model_manager)

# Helper to track loaded model
current_loaded_model_path = None

results = []

# Prepare mappings
sample_to_filename = df[['sample_id', 'filename']].drop_duplicates().set_index('sample_id')['filename'].to_dict()
# Get rank-0 token ID for the 'current' model for each sample
current_top1_map = df[(df['model'] == 'current') & (df['rank'] == 0)].set_index('sample_id')['token_id'].to_dict()
current_top1_str_map = df[(df['model'] == 'current') & (df['rank'] == 0)].set_index('sample_id')['token_str'].to_dict()


print(f"Processing {len(contrast_df)} samples for logit difference...")

for idx, row in tqdm(contrast_df.iterrows(), total=len(contrast_df), desc="Computing Logit Diffs"):
    sample_id = row['sample_id']
    contrast_token_id = row['contrast_token_id']
    extraction_source = row['extraction_source']
    
    # 1. Skip if no contrast token found
    if extraction_source == 'none' or pd.isna(contrast_token_id):
        results.append({
            'sample_id': sample_id,
            'logit_diff': None,
            'ref_logit': None,
            'contrast_logit': None,
            'ref_token': None,
            'ref_token_id': -1,
            'status': 'skipped_no_contrast'
        })
        continue

    # 2. Get necessary info
    filename = sample_to_filename.get(sample_id)
    if not filename:
        print(f"Skipping {sample_id}: filename not found")
        continue

    ref_token_id = current_top1_map.get(sample_id)
    ref_token_str = current_top1_str_map.get(sample_id)
    
    if ref_token_id is None:
        print(f"Skipping {sample_id}: ref_token_id not found")
        continue
        
    contrast_token_id = int(contrast_token_id)
    ref_token_id = int(ref_token_id)

    # 3. Load metadata
    file_path = traces_dir / filename
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        continue
        
    meta = data.get('metadata', {})
    model_path = meta.get('model')
    dtype_str = meta.get('dtype', 'torch.float16') 
    
    # Map dtype string to simple string expected by manager (if needed)
    # Manager expects: "float16", "bfloat16", "float32" or "auto"
    # The json has "torch.float16"
    if "float16" in dtype_str:
        dtype_arg = "bfloat16"
    elif "bfloat16" in dtype_str:
        dtype_arg = "bfloat16"
    elif "float32" in dtype_str:
        dtype_arg = "float32"
    else:
        dtype_arg = "auto"
    
    prompt = data.get('prompt', '')
    completion_in_prompt = data.get('completion_before_err', '')
    full_prompt = prompt + completion_in_prompt
    
    # 4. Load Model (Lazy Loading)
    if current_loaded_model_path != model_path:
        # Check if model string is valid
        if not model_path:
            print(f"Skipping {sample_id}: model path missing in metadata")
            continue
            
        try:
            model_manager.load_model(model_path=model_path, dtype=dtype_arg)
            current_loaded_model_path = model_path
        except Exception as e:
            print(f"Failed to load model {model_path}: {e}")
            continue

    # 5. Compute Logits
    try:
        # We assume compute_logits works as declared in core.py
        # returns: topk_data, last_logits, input_ids
        _, last_logits, _ = attribution_engine.compute_logits(full_prompt)
        
        # Ensure tensor is on CPU for value extraction
        last_logits = last_logits.cpu()
        
        ref_logit = last_logits[ref_token_id].item()
        contrast_logit = last_logits[contrast_token_id].item()
        logit_diff = ref_logit - contrast_logit
        
        results.append({
            'sample_id': sample_id,
            'ref_token': ref_token_str,
            'ref_token_id': int(ref_token_id),
            'ref_logit': ref_logit,
            'contrast_logit': contrast_logit,
            'logit_diff': logit_diff,
            'status': 'success'
        })
        
    except Exception as e:
        print(f"Error computing logits for {sample_id}: {e}")
        results.append({
            'sample_id': sample_id,
            'status': 'error',
            'error_msg': str(e)
        })

# Merge results
logit_results_df = pd.DataFrame(results)
final_df = contrast_df.merge(logit_results_df, on='sample_id', how='left')

print("Logit Difference Computation Complete.")
final_df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Plot the ECDF
plt.figure(figsize=(10,6))
sns.ecdfplot(data=final_df, x='logit_diff', label='Logit Difference ECDF')

# Calculate proportions for specific thresholds
thresholds = [1, 2, 3]
total_valid = final_df['logit_diff'].notna().sum()

for t in thresholds:
    prop = (final_df['logit_diff'] <= t).sum() / total_valid
    print(f"Proportion with logit diff <= {t}: {prop:.4f}")
    
    # Add vertical line
    plt.axvline(x=t, color='red', linestyle='--', alpha=0.5)
    # Add text annotation
    plt.text(t, 0.6, f'x={t}, p={prop:.2f}', color='red', ha='right', fontsize=9, rotation=90)

plt.title('CDF of Logit Differences (Ref - Contrast)')
plt.xlabel('Logit Difference')
plt.ylabel('Proportion')
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.legend()
plt.show()

In [ ]:
# Compute Logit Differences for Reference Models (1.7B and 4B)
ref_models = {
    "1p7b": "Qwen/Qwen3-1.7B",
    "4b": "Qwen/Qwen3-4B",
    "8b": "Qwen/Qwen3-8B",
    # "14b": "Qwen/Qwen3-14B"
}

# Tracker for model loading state
current_loaded_model_path = None
current_loaded_dtype = None

for label, target_model_path in ref_models.items():
    print(f"\nProcessing {len(final_df)} samples for {label} ({target_model_path}) logit difference...")
    results_ref = []
    
    for idx, row in tqdm(final_df.iterrows(), total=len(final_df), desc=f"Computing {label} Logits"):
        sample_id = row['sample_id']
        contrast_token_id = row['contrast_token_id']
        ref_token_id = row['ref_token_id']
        extraction_source = row['extraction_source']
        
        # 1. Skip checks
        if extraction_source == 'none' or pd.isna(contrast_token_id) or pd.isna(ref_token_id):
            results_ref.append({
                'sample_id': sample_id,
                f'logit_diff_{label}': None,
                f'ref_logit_{label}': None,
                f'contrast_logit_{label}': None,
                f'status_{label}': 'skipped'
            })
            continue

        # 2. Get filename
        filename = sample_to_filename.get(sample_id)
        if not filename:
            continue

        contrast_token_id = int(contrast_token_id)
        ref_token_id = int(ref_token_id)

        # 3. Read metadata for dtype and prompt
        file_path = traces_dir / filename
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            continue
        
        # Determine dtype from metadata
        meta = data.get('metadata', {})
        dtype_str = meta.get('dtype', 'torch.float16')
        
        if "float16" in dtype_str:
            dtype_arg = "bfloat16"
        elif "bfloat16" in dtype_str:
            dtype_arg = "bfloat16"
        elif "float32" in dtype_str:
            dtype_arg = "float32"
        else:
            dtype_arg = "auto"
            
        prompt = data.get('prompt', '')
        completion_in_prompt = data.get('completion_before_err', '')
        full_prompt = prompt + completion_in_prompt

        # 4. Load Model (Reload if changed)
        if current_loaded_model_path != target_model_path or current_loaded_dtype != dtype_arg:
            try:
                model_manager.load_model(model_path=target_model_path, dtype=dtype_arg)
                current_loaded_model_path = target_model_path
                current_loaded_dtype = dtype_arg
            except Exception as e:
                print(f"Failed to load {target_model_path}: {e}")
                current_loaded_model_path = None
                continue

        if current_loaded_model_path != target_model_path:
            continue

        # 5. Compute Logits
        try:
            _, last_logits, _ = attribution_engine.compute_logits(full_prompt)
            last_logits = last_logits.cpu()
            
            ref_logit = last_logits[ref_token_id].item()
            contrast_logit = last_logits[contrast_token_id].item()
            logit_diff = ref_logit - contrast_logit
            
            results_ref.append({
                'sample_id': sample_id,
                f'logit_diff_{label}': logit_diff,
                f'ref_logit_{label}': ref_logit,
                f'contrast_logit_{label}': contrast_logit,
                f'status_{label}': 'success'
            })
            
        except Exception as e:
            print(f"Error computing {label} logits for {sample_id}: {e}")
            results_ref.append({
                'sample_id': sample_id,
                f'status_{label}': 'error'
            })
    
    # Merge results
    print(f"Merging {label} results...")
    logit_results_df = pd.DataFrame(results_ref)
    
    # Remove columns if they already exist in final_df to avoid suffix hell if running multiple times
    for col in logit_results_df.columns:
        if col != 'sample_id' and col in final_df.columns:
            final_df.drop(columns=[col], inplace=True)
            
    final_df = final_df.merge(logit_results_df, on='sample_id', how='left')

print("Reference Models Logit Difference Computation Complete.")
final_df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import pandas as pd

# Define dynamic thresholds
roi_x_thresh = 2
roi_y_thresh = -2

# 1. Determine which columns are available for plotting
available_refs = []
if 'logit_diff_1p7b' in final_df.columns:
    available_refs.append(('logit_diff_1p7b', '1.7B'))
if 'logit_diff_4b' in final_df.columns:
    available_refs.append(('logit_diff_4b', '4B'))
if 'logit_diff_8b' in final_df.columns:
    available_refs.append(('logit_diff_8b', '8B'))
if 'logit_diff_14b' in final_df.columns:
    available_refs.append(('logit_diff_14b', '14B'))

if not available_refs:
    print("No reference logit difference columns found (logit_diff_1p7b, logit_diff_4b, logit_diff_8b).")
else:
    # Filter valid data for plotting (requiring current model logit_diff at minimum)
    plot_df = final_df.dropna(subset=['logit_diff']).copy()
    
    # Initialize highlight column to False
    plot_df['highlight'] = False
    
    num_plots = len(available_refs)
    fig, axes = plt.subplots(1, num_plots, figsize=(8 * num_plots, 8))
    
    if num_plots == 1:
        axes = [axes] # Make iterable
        
    for ax, (col_y, label) in zip(axes, available_refs):
        # Filter NaNs for this specific comparison
        # We create a mask for valid data in this comparison
        valid_mask = plot_df[col_y].notna()
        local_df = plot_df[valid_mask].copy()
        
        # Identify samples in the region of interest for this specific comparison
        # Region: Current > 2 AND Reference < -2
        local_highlight = (local_df['logit_diff'] > roi_x_thresh) & (local_df[col_y] < roi_y_thresh)
        
        # Update global highlight (Union) in the main plot_df
        # Use the index from local_highlight to update the corresponding rows in plot_df
        plot_df.loc[local_highlight.index, 'highlight'] = True
        
        roi_count = local_highlight.sum()
        
        print(f"[{label}] Plotting {len(local_df)} samples...")
        print(f"[{label}] Samples in highlighted region (x > {roi_x_thresh}, y < {roi_y_thresh}): {roi_count}")
        
        sns.scatterplot(
            data=local_df, 
            x='logit_diff', 
            y=col_y, 
            hue=local_highlight,
            palette={False: 'steelblue', True: 'red'},
            alpha=0.7,
            ax=ax
        )
        
        # Add diagonal line (y=x)
        all_vals = pd.concat([local_df['logit_diff'], local_df[col_y]])
        
        if not all_vals.empty:
            min_val, max_val = all_vals.min(), all_vals.max()
            buff = (max_val - min_val) * 0.05
            plot_min = min_val - buff
            plot_max = max_val + buff
            
            ax.plot([plot_min, plot_max], [plot_min, plot_max], 'r--', alpha=0.5, label='y=x')
            
            ax.axhline(0, color='gray', linestyle=':', alpha=0.6)
            ax.axvline(0, color='gray', linestyle=':', alpha=0.6)
            
            # Highlight region
            if plot_max > roi_x_thresh and plot_min < roi_y_thresh:
                rect_width = max(plot_max - roi_x_thresh, 0.1)
                rect_height = max(roi_y_thresh - plot_min, 0.1)
                
                rect = Rectangle(
                    (roi_x_thresh, plot_min),
                    width=rect_width,
                    height=rect_height, 
                    facecolor='orange', 
                    alpha=0.15, 
                    label=f'Region: x>{roi_x_thresh}, y<{roi_y_thresh}'
                )
                ax.add_patch(rect)
                
        ax.set_title(f'Logit Difference: Current vs {label}')
        ax.set_xlabel('Logit Difference (Current: 0.6B)')
        ax.set_ylabel(f'Logit Difference ({label})')
        # ax.set_xlim(-5, 30) # Optional: shared limits
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Correlation
        if len(local_df) > 1:
            corr = local_df['logit_diff'].corr(local_df[col_y])
            print(f"[{label}] Pearson Correlation: {corr:.4f}")
            ax.text(0.05, 0.95, f"r = {corr:.4f}", transform=ax.transAxes, 
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.show()
    
    print(f"Total Unique Highlighted Samples (Union): {plot_df['highlight'].sum()}")

In [ ]:
import json
import numpy as np

# 0. Setup Dynamic Path

# Infer Current Model Name
if 'plot_df' in locals() and not plot_df.empty:
    sample_0_file = traces_dir / sample_to_filename[plot_df.iloc[0]['sample_id']]
    with open(sample_0_file, 'r') as f:
        _meta = json.load(f).get('metadata', {})
        _model_path = _meta.get('model', 'UnknownModel')
        current_model_name = _model_path.split('/')[-1] # e.g. "Qwen3-0.6B"
else:
    current_model_name = "Qwen3-0.6B" # Default fallback
    _model_path = "Qwen/Qwen3-0.6B" # Assumption

# Define all models to process
# Format: label -> { path, role }
# role: 'current' is the main subject, 'ref' are comparisons
models_config = {
    current_model_name: {"path": _model_path, "role": "current"},
    "Qwen3-1.7B": {"path": "Qwen/Qwen3-1.7B", "role": "ref"},
    "Qwen3-4B": {"path": "Qwen/Qwen3-4B", "role": "ref"}
}

# Output Directory (New Structure: /exp/heatmaps/{bench_name}/)
base_output_dir = Path(f"/mnt/caiw_repos/AttnLRP/exp/heatmaps/{bench_name}")
print(f"Heatmap Base Directory: {base_output_dir}")

# 1. Identify Target Samples
# We use the highlight logic from 1.7B comparison primarily, or the Union logic if plot_df has it
target_samples_df = plot_df[plot_df['highlight']].copy()
print(f"Generating heatmaps for {len(target_samples_df)} samples...")

# Prepare Manifest Entry structure
manifest_entries = {} # sid -> data

# Helper to sanitize ID for filename (removes redundant folder levels)
def get_clean_id(sid, bench):
    sid = str(sid) # Cast to string to handle integer IDs safely
    prefix = f"{bench}/"
    if sid.startswith(prefix):
        return sid[len(prefix):]
    return sid.replace('/', '_')

# Initialize manifest entries with metadata
for _, row in target_samples_df.iterrows():
    sid = row['sample_id']
    # clean_id = get_clean_id(sid, bench_name)
    clean_id = sid
    
    # Collect available logit diffs
    diffs = {}
    if 'logit_diff' in row: diffs[current_model_name] = row['logit_diff']
    if 'logit_diff_1p7b' in row: diffs["Qwen3-1.7B"] = row['logit_diff_1p7b']
    if 'logit_diff_4b' in row: diffs["Qwen3-4B"] = row['logit_diff_4b']
    
    manifest_entries[sid] = {
        "sample_id": clean_id, # Use clean ID for viewer and listing
        "original_id": sid,    # Keep track of true ID
        "filename": sample_to_filename.get(sid, f"{sid}.json"),
        "ref_token": row['ref_token'],
        "contrast_token": row['contrast_token'],
        "logit_diffs": diffs
    }

# ---------------------------------------------------------
# Processing Loop
# ---------------------------------------------------------
current_loaded_model_path = None

for model_label, config in models_config.items():
    print(f"\n--- Processing Model: {model_label} ---")
    
    model_path = config['path']
    model_output_dir = base_output_dir / model_label
    model_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Pre-calculate list of items to process
    processing_list = []
    
    for sid in manifest_entries.keys():
        # Metadata check
        # For 'current', we read from trace to be sure of prompt etc.
        # For refs, we assume prompt is same as current trace
        
        # We need the prompt. Always read from the trace file associated with the sample
        trace_filename = sample_to_filename[sid]
        trace_path = traces_dir / trace_filename
        
        # Optimization: cache prompt reading?
        # For now, just read.
        with open(trace_path, 'r') as f:
            t_data = json.load(f)
            
        full_prompt = t_data.get('prompt', '') + t_data.get('completion_before_err', '')
        
        # IDs are constant across models for this analysis (by ref token strategy)
        # But we need to fetch them from our dataframe
        row = target_samples_df[target_samples_df['sample_id']==sid].iloc[0]
        
        processing_list.append({
            "sid": sid,
            "save_name": manifest_entries[sid]['sample_id'], # Use clean name for file
            "prompt": full_prompt,
            "ref_token_id": int(row['ref_token_id']),
            "contrast_token_id": int(row['contrast_token_id']),
        })
    
    # Load Model
    if current_loaded_model_path != model_path:
        try:
            print(f"Loading {model_path}...")
            # Unload previous to save memory if needed (optional implementation detail of manager)
            model_manager.load_model(model_path=model_path, dtype="bfloat16", lrp_rule="Attn-LRP")
            current_loaded_model_path = model_path
        except Exception as e:
            print(f"Failed to load {model_path}: {e}")
            continue
            
    # Compute and Save
    for item in tqdm(processing_list, desc=f"Attributing {model_label}"):
        sid = item['sid']
        save_name = item['save_name']
        out_file = model_output_dir / f"{save_name}.json"
        
        # Ensure parent directory exists? No, we flattened it.
        # But if save_name still has slashes (unexpected), we might need it.
        # out_file.parent.mkdir(parents=True, exist_ok=True)
        
        try:
            # 1. Logits
            _, _, input_tokens = attribution_engine.compute_logits(item['prompt'])
            
            # 2. Attribution
            bp_config = {
                "mode": "logit_diff",
                "strategy": "by_ref_token",
                "target_token_id": item['ref_token_id'],
                "ref_token_id": item['contrast_token_id'],
                "lrp_rule": "Attn-LRP"
            }
            
            relevance = attribution_engine.compute_input_attribution(bp_config)
            
            # 3. Construct Single Model Result
            result = {
                "meta": {
                    "sample_id": item['save_name'], # Use clean ID in file too
                    "original_id": sid,
                    "model": model_label,
                    "target_token_id": item['ref_token_id'],
                    "contrast_token_id": item['contrast_token_id']
                },
                "tokens": input_tokens,
                "relevance": relevance
            }
            
            # Clean floats
            def clean(obj):
                if isinstance(obj, list): return [clean(x) for x in obj]
                if isinstance(obj, dict): return {k: clean(v) for k, v in obj.items()}
                val = obj
                if hasattr(obj, 'item'): val = obj.item()
                if isinstance(val, (float, np.float32, np.float16, np.float64)):
                    if np.isnan(val) or np.isinf(val): return 0.0
                    return float(val)
                return val
            
            final_data = clean(result)
            
            with open(out_file, 'w') as f:
                json.dump(final_data, f, indent=2)
                
        except Exception as e:
            print(f"Error processing {sid} for {model_label}: {e}")

# Save Manifest
manifest_path = base_output_dir / "manifest.json"
manifest_data = {
    "models": list(models_config.keys()),
    "samples": list(manifest_entries.values())
}

# Values specific cleaning for manifest
def clean_manifest(obj):
    if isinstance(obj, list): return [clean_manifest(x) for x in obj]
    if isinstance(obj, dict): return {k: clean_manifest(v) for k, v in obj.items()}
    val = obj
    if hasattr(obj, 'item'): val = obj.item()
    if isinstance(val, (float, np.float32, np.float16, np.float64)):
        if np.isnan(val) or np.isinf(val): return None
        return float(val)
    return val

with open(manifest_path, 'w') as f:
    json.dump(clean_manifest(manifest_data), f, indent=2)

print(f"Saved manifest and data to {base_output_dir}")
print("Manifest models:", manifest_data['models'])

# ---------------------------------------------------------
# Viewer Instructions
# ---------------------------------------------------------
print("\nViewer Setup:")
print(f"Data is located in: {base_output_dir}")
print("To inspect results:")
print("1. Run HTTP Server in the heatmaps root:")
print(f"   cd /mnt/caiw_repos/AttnLRP/exp/heatmaps && python3 -m http.server 8180")
print("2. Open 'view.html' in your browser (e.g., http://localhost:8180/view.html)")


In [ ]:
from attnlrp_circuit.backend.circuit import CircuitAnalyzer
import networkx as nx

# Initialization
# model_manager = ModelManager()
# attribution_engine = AttributionEngine(model_manager)
circuit_analyzer = CircuitAnalyzer(attribution_engine)

# Output Base for Graphs
graph_output_base = Path("/mnt/caiw_repos/AttnLRP/exp/attribution_graphs") / bench_name
graph_output_base.mkdir(parents=True, exist_ok=True)
print(f"Graph Output Directory: {graph_output_base}")

# Helper to export graph
def export_graph_to_json(G, token_list=None):
    data = {"nodes": [], "links": []}
    if G is None: return data
    
    for n, attrs in G.nodes(data=True):
        # Node key strategy: "L{layer}_T{token}"
        # n is tuple (layer, token_idx)
        nid = f"L{n[0]}.T{n[1]}"
        
        node_data = {
            "id": nid,
            "layer": int(n[0]),
            "token": int(n[1]),
            "relevance": float(attrs.get('relevance', 0.0))
        }
        
        # Add string if available
        if token_list and n[1] < len(token_list):
            node_data["token_str"] = token_list[n[1]]
            
        data["nodes"].append(node_data)
        
    for u, v, attrs in G.edges(data=True):
        src_id = f"L{u[0]}.T{u[1]}"
        tgt_id = f"L{v[0]}.T{v[1]}"
        data["links"].append({
            "source": src_id,
            "target": tgt_id,
            "weight": float(attrs.get('weight', 0.0))
        })
    return data

# Process Graphs
current_loaded_model_path = None

for model_label, config in models_config.items():
    # if model_label != 'Qwen3-0.6B':
    #     continue

    print(f"\n--- Generating Graphs for Model: {model_label} ---")
    
    model_path = config['path']
    model_graph_dir = graph_output_base / model_label
    model_graph_dir.mkdir(parents=True, exist_ok=True)
    
    processing_list = []
    
    # Reuse manifest_entries from previous cell to target same filtered set
    for sid, entry in manifest_entries.items():
        # Filter Logic: Only process if in region (x > 2, y < -2)
        logit_diffs = entry.get('logit_diffs', {})
        cur_val = logit_diffs.get(current_model_name)
        
        # Filter
        if config['role'] == 'ref':
            ref_val = logit_diffs.get(model_label)
            if cur_val is None or ref_val is None: continue
            if not (cur_val > 2 and ref_val < -2):
                continue
        elif config['role'] == 'current':
            if cur_val is None or cur_val <= 2: continue
            # For current model, we keep it if it satisfies the criteria for AT LEAST ONE ref model
            # Since manifest_entries is already the filtered union, we just need to ensure cur > 2
        
        # Re-read Metadata/Prompt
        clean_id = entry['sample_id'] # Use clean ID from manifest
        trace_filename = entry['filename'] # From manifest
        trace_path = traces_dir / sample_to_filename.get(entry['original_id'], trace_filename)
        
        with open(trace_path, 'r') as f:
            t_data = json.load(f)
        
        full_prompt = t_data.get('prompt', '') + t_data.get('completion_before_err', '')
        
        row = target_samples_df[target_samples_df['sample_id'] == entry['original_id']].iloc[0]
        
        # Check for numpy types (int64) and convert to python native for JSON serialization
        sid_val = entry['original_id']
        if hasattr(sid_val, 'item'): sid_val = sid_val.item()
        
        save_name_val = clean_id
        if hasattr(save_name_val, 'item'): save_name_val = save_name_val.item()

        processing_list.append({
            "sid": sid_val,
            "save_name": save_name_val,
            "prompt": full_prompt,
            "ref_token_id": int(row['ref_token_id']),
            "contrast_token_id": int(row['contrast_token_id']),
        })

    # Load Model
    if current_loaded_model_path != model_path:
        try:
            print(f"Loading {model_path}...")
            model_manager.load_model(model_path=model_path, dtype="bfloat16", lrp_rule="Attn-LRP")
            current_loaded_model_path = model_path            
            # attribution_engine = AttributionEngine(model_manager)
            # circuit_analyzer = CircuitAnalyzer(attribution_engine)
        except Exception as e:
            print(f"Failed to load {model_path}: {e}")
            continue

    # Compute
    for item in tqdm(processing_list, desc=f"Graphing {model_label}"):

        # if item['sid'] != 301:
        #     continue

        sid = item['sid']
        save_name = item['save_name']
        out_file = model_graph_dir / f"{save_name}.json"
        
        try:
            # 1. Compute Logits (State Update)
            _, last_logits, input_tokens = attribution_engine.compute_logits(item['prompt'])
            # Ensure tensor is on CPU for value extraction
            last_logits = last_logits.cpu()
            
            ref_logit = last_logits[item['ref_token_id']].item()
            contrast_logit = last_logits[item['contrast_token_id']].item()
            logit_diff = ref_logit - contrast_logit
            print("logit_diff = ", logit_diff)
            
            # 2. Config
            bp_config = {
                "mode": "logit_diff",
                "strategy": "by_ref_token",
                "target_token_id": item['ref_token_id'],
                "ref_token_id": item['contrast_token_id'],
                "lrp_rule": "Attn-LRP"
            }
            
            # 3. Compute Matrices
            connection_data = circuit_analyzer.compute_connection_matrices(bp_config)
            
            # 4. Build Graph (Pruning)
            G_full, _ = circuit_analyzer.build_graph_from_matrices(
                connection_data, 
                pruning_mode="by_per_layer_cum_mass_percentile",
                top_p=0.9
            )
            
            # 5. Get Connected Subgraph
            G_sub, _ = circuit_analyzer.get_connected_subgraph(G_full, target_node=None)
            
            # 6. Export
            json_data = {
                "meta": {
                    "sample_id": save_name,
                    "original_id": sid,
                    "model": model_label,
                    "target_token_id": item['ref_token_id'],
                    "contrast_token_id": item['contrast_token_id']
                },
                "tokens": input_tokens, # Save full list just in case
                "graph": export_graph_to_json(G_sub, token_list=input_tokens)
            }
            
            with open(out_file, 'w') as f:
                json.dump(json_data, f, indent=2)
                
        except Exception as e:
            print(f"Error processing graph for {sid}: {e}")
            with open(out_file, 'w') as f:
                json.dump({"error": str(e), "sid": sid}, f)

# Copy Manifest
import shutil
src_manifest = base_output_dir / "manifest.json"
dst_manifest = graph_output_base / "manifest.json"
if src_manifest.exists():
    shutil.copy(src_manifest, dst_manifest)

print("\nGraph Generation Complete.")